### 1. Dataset Overview (Dataset: Diabetes)

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
df = data.frame

print(f"Missing Values\n{df.isnull().sum()}\n")

print(f"Datatype Of Columns\n{df.dtypes}\n")

### 2. Correlation Heatmap 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

numeric_data = df.select_dtypes(include="number")

correlation = numeric_data.corr()

sns.heatmap(
    data=correlation,
    annot=True,
    cmap="coolwarm"
)
plt.show()

### 3. ColumnTransformer and Pipeline

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

data = load_diabetes(as_frame=True)

X = data.data
y = data.target

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_cols = list(X.columns)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
])

prep_pipeline = Pipeline([
    ("preprocessor", preprocessor),
])

X_train_processed = prep_pipeline.fit_transform(x_train)
X_test_processed = prep_pipeline.transform(x_test)

feature_names = prep_pipeline.named_steps["preprocessor"].get_feature_names_out()
print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("\nFeature names:", feature_names)
print("Total features:", len(feature_names))

### 4. Model Creation (LinearRegression)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])

model.fit(x_train, y_train)

y_pred = model.predict(x_test)

mse = mean_squared_error(y_test, y_pred)
print("MAE :", round(mean_absolute_error(y_test, y_pred), 2))
print("MSE :", round(mse, 2))
print("RMSE:", round(np.sqrt(mse), 2))
print("R2  :", round(r2_score(y_test, y_pred), 3))
print("Train R2:", round(model.score(x_train, y_train), 3))